# 电影圈数据

In [1]:
import pandas as pd
import numpy as np
import networkx as nx

In [2]:
import sys
sys.path.append("..")

# 数据处理

## 原始数据

In [3]:
df_raw = pd.read_csv("../data/cast/dwd_cast_works.csv")
df_raw.head()

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,k_genres,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,喜剧/动作,无评分,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,爱情,无评分,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓
4,1013885,阿美莉嘉·奥利沃,女,10727641,演员,是,21,10727641,碟中谍5：神秘国度,电影,...,动作/惊悚/冒险,有评分,7.8,292193,292193.0,1510.0,21455331,2027819,5,Turandot


## 数据筛选
过滤条件
1. 电影
2. 年份为2020年及之后
3. 有评分
4. 地区含中国

In [6]:
df_movie = df_raw[
    (df_raw['k_type'] == '电影') &
    (df_raw['k_movie_year'] >= 2020) & 
    (df_raw['is_rating'] == '有评分') & 
    (df_raw['k_region'].str.contains('中国'))]
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,k_genres,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray
205,27553997,杨大鹏,男,10604086,演员,是,51,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,55108043,206,惧留孙
228,27488698,赵磊,男,10604086,演员,是,52,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,54977445,229,普贤真人
344,27481220,杨立新,男,10604086,演员,是,20,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,54962489,345,东伯侯姜桓楚
977,30094565,曹升,男,10604086,编剧,是,4,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,60189179,978,NaN
1010,27216195,袁泉,女,10604086,演员,是,10,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,54432439,1011,姜王后
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602394,27484950,张承,男,35155748,演员,否,999,35155748,金刚川,电影,...,剧情/战争,有评分,6.5,259405,259405.0,1299.0,70311545,54969949,602395,文人
602422,27576108,付赫安琪,女,35027719,演员,是,3,35027719,兴安岭猎人传说,电影,...,悬疑/恐怖/冒险,有评分,6.0,24286,24286.0,382.0,70055487,55152265,602423,疯女人
602434,30372558,郭阳,男,37096787,演员,是,12,37096787,真爱营业,电影,...,喜剧/爱情,有评分,5.1,3934,3934.0,142.0,74193623,60745165,602435,儒雅弟弟
602463,27483250,张天其,男,35496803,演员,是,3,35496803,河豚,电影,...,剧情,有评分,4.9,6237,6237.0,175.0,70993655,54966549,602464,NaN


In [8]:
# 仅保留演员，导演，制片人数据
df_movie = df_movie[df_movie['k_role'].isin(['演员', '导演', '制片人'])]
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,k_genres,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray
205,27553997,杨大鹏,男,10604086,演员,是,51,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,55108043,206,惧留孙
228,27488698,赵磊,男,10604086,演员,是,52,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,54977445,229,普贤真人
344,27481220,杨立新,男,10604086,演员,是,20,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,54962489,345,东伯侯姜桓楚
1010,27216195,袁泉,女,10604086,演员,是,10,10604086,封神第一部：朝歌风云,电影,...,动作/战争/奇幻/古装,有评分,7.8,1141380,1141380.0,2984.0,21208221,54432439,1011,姜王后
1027,27485568,曹炳琨,男,26816376,演员,是,3,26816376,超级的我,电影,...,奇幻/冒险,有评分,4.8,11299,11299.0,233.0,53632801,54971185,1028,三哥
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602339,27568621,欧豪,男,26754233,演员,是,6,26754233,八佰,电影,...,剧情/历史/战争,有评分,7.5,761649,761649.0,2390.0,53508515,55137291,602340,端午
602394,27484950,张承,男,35155748,演员,否,999,35155748,金刚川,电影,...,剧情/战争,有评分,6.5,259405,259405.0,1299.0,70311545,54969949,602395,文人
602422,27576108,付赫安琪,女,35027719,演员,是,3,35027719,兴安岭猎人传说,电影,...,悬疑/恐怖/冒险,有评分,6.0,24286,24286.0,382.0,70055487,55152265,602423,疯女人
602434,30372558,郭阳,男,37096787,演员,是,12,37096787,真爱营业,电影,...,喜剧/爱情,有评分,5.1,3934,3934.0,142.0,74193623,60745165,602435,儒雅弟弟


In [10]:
# 添加新的movie_id_m列
df_movie = df_movie.copy()
df_movie['movie_id_m'] = df_movie['k_movie_id'].apply(lambda x: 'm' + str(x))
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m
205,27553997,杨大鹏,男,10604086,演员,是,51,10604086,封神第一部：朝歌风云,电影,...,有评分,7.8,1141380,1141380.0,2984.0,21208221,55108043,206,惧留孙,m21208221
228,27488698,赵磊,男,10604086,演员,是,52,10604086,封神第一部：朝歌风云,电影,...,有评分,7.8,1141380,1141380.0,2984.0,21208221,54977445,229,普贤真人,m21208221
344,27481220,杨立新,男,10604086,演员,是,20,10604086,封神第一部：朝歌风云,电影,...,有评分,7.8,1141380,1141380.0,2984.0,21208221,54962489,345,东伯侯姜桓楚,m21208221
1010,27216195,袁泉,女,10604086,演员,是,10,10604086,封神第一部：朝歌风云,电影,...,有评分,7.8,1141380,1141380.0,2984.0,21208221,54432439,1011,姜王后,m21208221
1027,27485568,曹炳琨,男,26816376,演员,是,3,26816376,超级的我,电影,...,有评分,4.8,11299,11299.0,233.0,53632801,54971185,1028,三哥,m53632801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602339,27568621,欧豪,男,26754233,演员,是,6,26754233,八佰,电影,...,有评分,7.5,761649,761649.0,2390.0,53508515,55137291,602340,端午,m53508515
602394,27484950,张承,男,35155748,演员,否,999,35155748,金刚川,电影,...,有评分,6.5,259405,259405.0,1299.0,70311545,54969949,602395,文人,m70311545
602422,27576108,付赫安琪,女,35027719,演员,是,3,35027719,兴安岭猎人传说,电影,...,有评分,6.0,24286,24286.0,382.0,70055487,55152265,602423,疯女人,m70055487
602434,30372558,郭阳,男,37096787,演员,是,12,37096787,真爱营业,电影,...,有评分,5.1,3934,3934.0,142.0,74193623,60745165,602435,儒雅弟弟,m74193623


## 影人职责合并

In [13]:
# 合并影人职责
df_cast = df_movie[['k_cast_id', 'cast_name', 'k_role']].drop_duplicates()
df_cast_agg = df_cast.groupby(['k_cast_id', 'cast_name'])['k_role'].apply(lambda x: '/'.join(sorted(x.unique()))).reset_index()
df_cast_agg

,k_cast_id,cast_name,k_role
0,2003387,陈哲艺,导演
1,2012019,赵涛,演员
2,2019839,池松壮亮,演员
3,2021201,罗伯特·克耐普,演员
4,2029977,黄炳耀,导演
...,...,...,...
6663,74888459,马库斯·扬·哈曼,演员
6664,74888461,孟响,演员
6665,74888463,孟宪访,演员
6666,74888465,李倩,制片人


In [17]:
df_cast_agg[df_cast_agg['cast_name'] == '张艺谋']

,k_cast_id,cast_name,k_role
1787,54520381,张艺谋,导演/演员


In [19]:
df_movie['cast_role_agg'] = df_movie['k_cast_id'].map(
    df_cast_agg.set_index('k_cast_id')['k_role']
)
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
205,27553997,杨大鹏,男,10604086,演员,是,51,10604086,封神第一部：朝歌风云,电影,...,7.8,1141380,1141380.0,2984.0,21208221,55108043,206,惧留孙,m21208221,演员
228,27488698,赵磊,男,10604086,演员,是,52,10604086,封神第一部：朝歌风云,电影,...,7.8,1141380,1141380.0,2984.0,21208221,54977445,229,普贤真人,m21208221,演员
344,27481220,杨立新,男,10604086,演员,是,20,10604086,封神第一部：朝歌风云,电影,...,7.8,1141380,1141380.0,2984.0,21208221,54962489,345,东伯侯姜桓楚,m21208221,演员
1010,27216195,袁泉,女,10604086,演员,是,10,10604086,封神第一部：朝歌风云,电影,...,7.8,1141380,1141380.0,2984.0,21208221,54432439,1011,姜王后,m21208221,演员
1027,27485568,曹炳琨,男,26816376,演员,是,3,26816376,超级的我,电影,...,4.8,11299,11299.0,233.0,53632801,54971185,1028,三哥,m53632801,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602339,27568621,欧豪,男,26754233,演员,是,6,26754233,八佰,电影,...,7.5,761649,761649.0,2390.0,53508515,55137291,602340,端午,m53508515,演员
602394,27484950,张承,男,35155748,演员,否,999,35155748,金刚川,电影,...,6.5,259405,259405.0,1299.0,70311545,54969949,602395,文人,m70311545,演员
602422,27576108,付赫安琪,女,35027719,演员,是,3,35027719,兴安岭猎人传说,电影,...,6.0,24286,24286.0,382.0,70055487,55152265,602423,疯女人,m70055487,演员
602434,30372558,郭阳,男,37096787,演员,是,12,37096787,真爱营业,电影,...,5.1,3934,3934.0,142.0,74193623,60745165,602435,儒雅弟弟,m74193623,演员


In [20]:
df_movie[df_movie['cast_name'] == '张艺谋']

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
8970,27260166,张艺谋,男,35294995,演员,是,40,35294995,我和我的父辈,电影,...,6.9,175739,175739.0,1101.0,70590039,54520381,8971,电视台台长,m70590039,导演/演员
130379,27260166,张艺谋,男,35215390,导演,是,1,35215390,狙击手,电影,...,7.7,340264,340264.0,1619.0,70430829,54520381,130380,NaN,m70430829,导演/演员
130391,27260166,张艺谋,男,33447633,导演,是,1,33447633,坚如磐石,电影,...,6.0,344998,344998.0,1439.0,66895315,54520381,130392,NaN,m66895315,导演/演员
200266,27260166,张艺谋,男,30257787,导演,是,1,30257787,一秒钟,电影,...,7.7,221244,221244.0,1305.0,60515623,54520381,200267,NaN,m60515623,导演/演员
491105,27260166,张艺谋,男,35766491,导演,是,1,35766491,满江红,电影,...,7.0,1081340,1081340.0,2751.0,71533031,54520381,491106,NaN,m71533031,导演/演员
561241,27260166,张艺谋,男,36208094,导演,是,1,36208094,第二十条,电影,...,7.5,766017,766017.0,2397.0,72416237,54520381,561242,NaN,m72416237,导演/演员


# 图对象

In [21]:
G = nx.from_pandas_edgelist(
    df_movie,
    source='k_cast_id',
    target='movie_id_m',
    edge_attr=True,
    create_using=nx.Graph()
)
G.number_of_nodes(), G.number_of_edges()

(7735, 12246)

In [22]:
# 子图数量
len(list(nx.connected_components(G)))

93

In [23]:
# 最大联通分支
largest_cc = max(nx.connected_components(G), key=len)
G_largest = G.subgraph(largest_cc).copy()
G_largest.number_of_nodes(), G_largest.number_of_edges()

(7311, 11900)

## 绘图